# Rust-backed diagnostics quickstart
Use the lightweight `basic` recorder to capture convergence history and Fourier coverage, then render the two overview plots inline.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import fpm_rs as fpm

In [ ]:
optics = fpm.Optics(532e-9, 0.10, 4.0, 6.5e-6)
illumination = fpm.LEDArray((3, 3), 4e-3, 90e-3, (1.0, 1.0))
model = fpm.compile_model(optics, illumination, (16, 16), (32, 32))

row, column = np.indices(model.reconstruction_shape)
amplitude = 0.7 + 0.3 * ((row // 4 + column // 4) % 2)
phase = 0.3 * np.sin(row / 5.0) * np.cos(column / 7.0)
object_field = np.asarray(amplitude * np.exp(1j * phase), dtype=np.complex128)

simulation = fpm.simulate(model, object_field, seed=17)
problem = fpm.ReconstructionProblem(
    simulation.measurements,
    simulation.reconstruction_model,
    name="diagnostics-quickstart",
)
recorder = fpm.DiagnosticRecorder("basic")
result = fpm.AlternatingProjection(iterations=6).run(
    problem,
    callbacks=[recorder],
)
diagnostics = recorder.diagnostics()
result.runtime.completed_iterations, result.final_loss

In [ ]:
diagnostics["iteration_history"][-1]

In [ ]:
plot_results = {
    "convergence": fpm.plot.plot_convergence(diagnostics),
    "fourier_coverage": fpm.plot.plot_fourier_coverage(diagnostics),
}
for plot_name, plot in plot_results.items():
    if plot is None:
        continue
    figure, axes = plot
    display(figure)
    # figure.savefig(f"{plot_name}.png", dpi=180, bbox_inches="tight")
    plt.close(figure)